# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks, then compute the State of Nature Score per the State of Nature Module PRD v2.0.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Configure and run Tier 1 (regional) + Tier 2 (contemporary) benchmarking
5. View the benchmark scorecard
6. Compute State of Nature concern levels and SoN Score
7. Download results

**Reference benchmarking methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection (adapted)
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

**State of Nature scoring methodology:**
- Darukaa State of Nature Module PRD v2.0
- TNFD LEAP Annex 2 (Ecosystem Extent / Condition / Species Population / Extinction Risk)
- Normalized sum formula: `SoN_Score = (Σ dim_scores − 4) / 16 × 10`

## 1. Setup — Clone repo & install

In [ ]:
# ── Setup: Clone or update repo ──────────────────────────
import os
os.chdir("/content")  # Always start from a safe location

REPO_DIR = "/content/reference-benchmarking"
CODE_DIR = f"{REPO_DIR}/darukaa_reference_v0.1.0"

if os.path.exists(REPO_DIR):
    os.chdir(CODE_DIR)
    !git pull origin main
    print("✓ Pulled latest changes")
else:
    !git clone https://@github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}
    os.chdir(CODE_DIR)
    print("✓ Cloned fresh")

!pip install -e . --force-reinstall -q 2>&1 | tail -1

import sys
cleared = [k for k in sys.modules if 'darukaa' in k]
for m in cleared:
    del sys.modules[m]

os.chdir("/content")
print(f"✓ Installed from {CODE_DIR}, cleared {len(cleared)} cached modules")

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 11 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 4.12 KiB | 124.00 KiB/s, done.
From https://github.com/G-auravSingh/reference-benchmarking
 * branch            main       -> FETCH_HEAD
   871956b..092af24  main       -> origin/main
Updating 871956b..092af24
Fast-forward
 README.md                                               |  5 +++++
 darukaa_reference_v0.1.0/README.md                      |  5 +++++
 darukaa_reference_v0.1.0/darukaa_reference/reference.py | 11 ++++++++++-
 3 files changed, 20 insertions(+), 1 deletion(-)
✓ Pulled latest changes
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
✓ Installed from /content/reference-benchmarking/darukaa_reference_v0.1.0, cle

## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [ ]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "gaurav-singh-007"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

✓ GEE initialised with project: gaurav-singh-007


## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [ ]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

Upload your KML/KMZ/GeoJSON file:


Saving Site 3 Karpa_kml.kml to Site 3 Karpa_kml (1).kml

✓ Uploaded: Site 3 Karpa_kml (1).kml (3.0 KB)






## 4. Configure the pipeline

Key parameters:
- `HMI_CEILING`: 0.10 (adapted from SEED's 0.05 for buffer-scale analysis — see README)
- `ELEVATION_BAND_M`: ±300m elevation stratification for Tier 2
- `ENABLED_INDICATORS`: leave empty `[]` for all 44 registered indicators
- `EDNA_POINTS_ASSET`: optional GEE FeatureCollection path for HSAS true scoring

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# ── Project type ──────────────────────────────────────────────────────────────
# "conservation" : single KML/KMZ site → runs exactly as before
# "agroforestry" : multiple cluster KMLs + manifest.json → aggregates internally
PROJECT_TYPE = "conservation"  # @param ["conservation", "agroforestry"]

# ── Trajectory tracking ───────────────────────────────────────────────────────
# IS_BASELINE_RUN = True  → Year 0. Saves baseline_scorecard.csv; no trajectory cols.
# IS_BASELINE_RUN = False → Year 1+. Loads baseline_scorecard.csv and computes Δ.
IS_BASELINE_RUN = True  # @param {type:"boolean"}

# For agroforestry only: path to manifest JSON with per-cluster area weights.
# Format: reference_clusters_manifest.json from the generalised site selection pipeline.
# Top-level keys: project_name, clusters (array), total_area_ha.
# Each cluster: {"cluster_label": "RC1", "kml_file": "/content/reference_cluster_RC1.kml", "area_ha": 20.04, ...}
# Leave as None for conservation projects.
MANIFEST_FILE = "/content/reference_clusters_manifest.json"  # e.g. "/content/manifest.json"

# Baseline file path (used for trajectory on Year 1+ runs)
BASELINE_FILE = "./output/baseline_scorecard.csv"

# ── Which indicators to run? Leave empty [] for all 44. ──────────────────────
# v3.0 indicators available:
#   Dim 1 (Extent):     natural_habitat, natural_landcover, cpland, forest_loss_rate, kba_overlap
#   Dim 2 (Condition):  ndvi, habitat_health, flii, eii, eii_structural, eii_compositional,
#                       eii_functional, bii, pdf, aridity_index,
#                       tspi, sabf, wcpi, wsdi, hsas, edpp, mspl, rci,   ← aquatic/eDNA
#                       riparian_ndvi_trend, jrc_water_persistence, shdi, lai, chm  ← v6.5
#   Dim 3 (Population): endemic_richness, flagship_habitat, endemic_plant_richness
#   Dim 4 (Extinction): threatened_richness, ceri, star_t, threatened_plant_richness
#   Threats:            ghm, light_pollution, hdi, lst_day, lst_night,
#                       sdi, stsi, iri, ivsi
ENABLED_INDICATORS = []  # e.g. ["ndvi", "bii", "eii"] to run a subset

# ── Tier 2 reference selection parameters ─────────────────────────────────────
HMI_CEILING = 0.2   #use 0.3 for agroforestry  # Adapted from SEED's 0.05 for buffer-scale analysis
ELEVATION_BAND_M = 300.0 # ±m elevation filter for Tier 2 stratification
MIN_REF_PIXELS = 5       # SEED minimum before fallback cascade

# ── Remote sensing year ───────────────────────────────────────────────────────
NDVI_YEAR = 2025  # @param {type:"integer"}
LST_YEAR  = 2025  # @param {type:"integer"}

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "./output"

# ─────────────────────────────────────────────────────────────────────────────
print(f"Project type:        {PROJECT_TYPE}")
print(f"Baseline run:        {IS_BASELINE_RUN}")
print(f"HMI ceiling:         {HMI_CEILING}")
print(f"Elevation band:      ±{ELEVATION_BAND_M}m")
print(f"Min ref pixels:      {MIN_REF_PIXELS}")
print(f"Remote sensing year: {NDVI_YEAR}")
print(f"Indicators:          {'all 44' if not ENABLED_INDICATORS else ENABLED_INDICATORS}")
print(f"eDNA asset:          {EDNA_POINTS_ASSET or 'not set (HSAS returns habitat suitability only)'}")
if PROJECT_TYPE == "agroforestry":
    print(f"Manifest file:       {MANIFEST_FILE}")

Project type:        conservation
Baseline run:        True
HMI ceiling:         0.2
Elevation band:      ±300.0m
Min ref pixels:      5
Remote sensing year: 2025
Indicators:          all 25


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Run the pipeline

In [ ]:
import logging, os, json as _json
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
import sys
sys.path.append("/content/reference-benchmarking/darukaa_reference_v0.1.0")
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Build shared config ────────────────────────────────────────────────────────
raster_paths = {
    "iucn_mammals": "projects/darukaa-earth130226/assets/RedList_Mammals_Terrestrial",
    "iucn_birds":   "projects/darukaa-earth130226/assets/RedList_Bird_IUCN_Category",
    "kba_global":   "projects/darukaa-earth130226/assets/KBA_Global_POL_SEP25",
    "pv_binary":    "projects/darukaa-earth-product/assets/biodiversity_India_PV_Binary_2025_Full_Mosaic",
    "msa":          "projects/ee-jayankandir/assets/TerrestrialMSA_2015_World",
}
# Add eDNA points asset if provided (enables HSAS true scoring)
if EDNA_POINTS_ASSET:
    raster_paths["edna_points_asset"] = EDNA_POINTS_ASSET

config = Config(
    gee_project=GEE_PROJECT,
    bii_gee_asset="projects/gaurav-singh-007/assets/bii-2020_v2-1-1",
    hmi_hard_ceiling=HMI_CEILING,
    elevation_band_m=ELEVATION_BAND_M,
    min_reference_pixels=MIN_REF_PIXELS,
    ndvi_year=NDVI_YEAR,
    lst_year=LST_YEAR,
    raster_paths=raster_paths,
    output_dir=OUTPUT_DIR,
    output_format="both",
    enabled_indicators=ENABLED_INDICATORS,
)

registry = create_default_registry()
print(f"✓ Registry loaded: {len(registry)} indicators")
t2 = registry.tier2_indicators()
print(f"  Tier 2 eligible: {len(t2)} indicators")
print(f"  Tier 2 excluded: {len(registry)-len(t2)} indicators (threats/aquatic/scalar/Protocol-A)")

# ── Run pipeline — conservation (single site) or agroforestry (cluster loop) ──
if PROJECT_TYPE == "conservation":
    pipeline = Pipeline(config, registry)
    report   = pipeline.run(
        site_path=SITE_FILE,
        output_path=f"{OUTPUT_DIR}/benchmark_scorecard",
    )
    print(f"\n✓ Pipeline complete")

elif PROJECT_TYPE == "agroforestry":
    if MANIFEST_FILE is None:
        raise ValueError("MANIFEST_FILE must be set for agroforestry projects.")
    with open(MANIFEST_FILE) as _f:
        manifest_json = _json.load(_f)
    manifest = manifest_json["clusters"]
    print(f"Manifest loaded: {len(manifest)} clusters ({manifest_json.get('project_name','')})")

    for entry in manifest:
        if not os.path.exists(entry["kml_file"]):
            raise FileNotFoundError(f"Cluster KML not found: {entry['kml_file']}")

    cluster_scorecards = []
    total_area = manifest_json.get("total_area_ha") or sum(e["area_ha"] for e in manifest)

    for i, entry in enumerate(manifest):
        cid   = entry["cluster_label"]
        kpath = entry["kml_file"]
        area  = entry["area_ha"]
        print(f"\n  [{i+1}/{len(manifest)}] Running cluster: {cid} ({area:.1f} ha) ...")
        pipeline = Pipeline(config, registry)
        r = pipeline.run(site_path=kpath, output_path=f"{OUTPUT_DIR}/benchmark_scorecard_{cid}")
        cluster_scorecards.append((r["scorecard"], area))
        print(f"  ✓ {cid} complete — {len(r['scorecard'])} indicators")

    import pandas as pd, numpy as np
    print(f"\nAggregating {len(manifest)} clusters (total {total_area:.1f} ha)...")

    frames = []
    for rows, area in cluster_scorecards:
        cdf = pd.DataFrame(rows); cdf["_area"] = area; frames.append(cdf)
    all_df = pd.concat(frames, ignore_index=True)

    def weighted_mean(vals, weights):
        vals = np.array(vals, dtype=float); weights = np.array(weights, dtype=float)
        mask = ~np.isnan(vals)
        if mask.sum() == 0: return np.nan
        return np.average(vals[mask], weights=weights[mask])

    aggregated_rows = []
    for ind_name, grp in all_df.groupby("display_name", sort=False):
        areas = grp["_area"].values
        sv  = weighted_mean(grp["site_value"].values, areas)
        t1r = weighted_mean(grp["tier1_reference"].values, areas)
        t2r = weighted_mean(grp["tier2_reference"].values, areas)
        def parse_int(x):
            if isinstance(x, float): return x
            if isinstance(x, str) and x.endswith("%"):
                try: return float(x.strip("%"))/100
                except: return np.nan
            try: return float(x)
            except: return np.nan
        t1i = weighted_mean([parse_int(v) for v in grp["tier1_intactness"].values], areas)
        t2i = weighted_mean([parse_int(v) for v in grp["tier2_intactness"].values], areas)
        n_t2 = grp["tier2_reference"].notna().sum()
        agg_row = dict(grp.iloc[0])
        agg_row.update({
            "site_id": "project_aggregated",
            "site_value": sv if not np.isnan(sv) else None,
            "tier1_reference": t1r if not np.isnan(t1r) else None,
            "tier2_reference": t2r if not np.isnan(t2r) else None,
            "tier1_intactness": t1i if not np.isnan(t1i) else None,
            "tier2_intactness": t2i if not np.isnan(t2i) else None,
            "n_clusters_total": len(manifest),
            "n_clusters_with_data": int((~np.isnan(grp["site_value"].astype(float))).sum()),
            "n_clusters_tier2": int(n_t2),
            "total_area_ha": total_area,
            "aggregation_note": f"Area-weighted mean across {len(manifest)} DBSCAN clusters",
            "_area": total_area,
        })
        aggregated_rows.append(agg_row)

    report = {
        "scorecard": aggregated_rows,
        "project_type": "agroforestry",
        "n_clusters": len(manifest),
        "total_area_ha": total_area,
    }
    print(f"✓ Aggregation complete: {len(aggregated_rows)} indicators, {len(manifest)} clusters")

else:
    raise ValueError(f"Unknown PROJECT_TYPE: {PROJECT_TYPE!r}. Use 'conservation' or 'agroforestry'.")

✓ Registry loaded: 25 indicators
  Tier 2 eligible: 13 indicators
  Tier 2 excluded: 12 indicators (threats/protocol-A)



✓ Pipeline complete


## 6. Benchmark Scorecard

Raw output: site values, Tier 1 (regional) and Tier 2 (contemporary least-disturbed) references, and intactness ratios for all indicators.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(report["scorecard"])

_extra_cols = ["n_clusters_total", "n_clusters_with_data", "n_clusters_tier2",
               "total_area_ha", "aggregation_note"]
_agg_cols = [c for c in _extra_cols if c in df.columns]
if _agg_cols:
    print(f"Project type: agroforestry — {report.get('n_clusters','')} clusters, "
          f"{report.get('total_area_ha',''):.1f} ha total")

display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

df_display = df[existing_cols].copy()
for col in ["site_value", "tier1_reference", "tier2_reference"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.4f}" if pd.notna(x) else "—"
        )
for col in ["tier1_intactness", "tier2_intactness"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.1%}" if pd.notna(x) else "—"
        )

# ── Protocol A indicator names (assessed via absolute threshold) ─────────────
PROTOCOL_A_NAMES = {
    "Composite Extinction-Risk Index", "STAR_T (Threat Abatement)",
    "KBA/IBA Overlap", "Flagship Habitat Viability",
    "Endemic Species Richness", "Threatened Species Richness",
    "Endemic Plant Species Richness", "Threatened Plant Species Richness",
    "Biodiversity Intactness Index", "Forest Landscape Integrity Index",
}

# ── Threat indicator names (pillar 5, contextual only) ───────────────────────
THREATS_NAMES = {
    "Global Human Modification", "Light Pollution (VIIRS)",
    "Human Disturbance Index", "Daytime Surface Temperature",
    "Nighttime Surface Temperature",
    # v6.4 aquatic threats
    "Shoreline Disturbance Index", "Surface Temperature Stress Index",
    "Invasive Risk Index",
    # v6.5
    "Invasive Vegetation Spread Index",
}

# ── Aquatic/eDNA/scalar indicators — note-worthy interpretation ──────────────
AQUATIC_NOTE_NAMES = {
    "Trophic State Proxy Index (NDCI)", "Surface Algal Bloom Frequency",
    "Water Clarity Proxy Index", "Water Surface Dynamics Index",
    "Habitat Suitability Alignment Score", "eDNA Persistence Potential",
    "Microbial Stress Probability Layer", "Riparian Complexity Index",
    "Riparian NDVI Temporal Trend", "JRC Water Persistence (Permanent Fraction)",
    "Shoreline Development Index (Morphometric)",
}

def _fix_interpretation(row):
    interp = row.get("interpretation", "") or ""
    if "Insufficient data" not in interp and interp not in ("", "—"):
        return interp

    sv    = row.get("site_value")
    t1    = row.get("tier1_reference")
    t1i   = row.get("tier1_intactness")
    t2i   = row.get("tier2_intactness")
    dname = row.get("display_name", "")

    if dname in PROTOCOL_A_NAMES:
        if sv is not None and str(sv) not in ("", "nan", "None"):
            return "Assessed via Protocol A absolute threshold (no spatial reference)"
        return "No site value returned — check GEE asset coverage or site polygon size"

    if dname in THREATS_NAMES:
        if sv is not None and str(sv) not in ("", "nan", "None"):
            return "Contextual pressure indicator — see threat assessment below"
        return "No site value returned"

    if dname in AQUATIC_NOTE_NAMES:
        if sv is not None and str(sv) not in ("", "nan", "None"):
            return "Aquatic/riparian metric — site-specific, no Tier 2 reference"
        return "No site value — aquatic metric requires water body in site polygon"

    if dname == "Landscape Connectivity (CPLAND)":
        return "Assessed via site value only — Tier 1 ratio not applicable for binary asset"

    if t1 is not None and float(t1) == 0.0 and sv is not None:
        return "Regional reference = 0 (no natural land cover in MODIS buffer) — ratio undefined"

    if t1i is not None and t2i is None:
        return f"Tier 2 unavailable — Tier 1 only ({float(t1i):.1%} of regional reference)"

    if t1i is None and t2i is None:
        return "No reference computable — check site polygon size or GEE asset coverage"

    return interp

if "interpretation" in df.columns:
    df["interpretation"] = df.apply(_fix_interpretation, axis=1)
    if "interpretation" in df_display.columns:
        df_display["interpretation"] = df["interpretation"]

pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 60)
display(df_display)

BENCHMARK SCORECARD: 25 indicator-site pairs



,site_id,display_name,pillar,site_value,tier1_reference,tier1_intactness,tier2_reference,tier2_intactness,interpretation
0,Site 3 Karpa_kml (1)_0000,Natural Habitat Extent,1,97.0667,100.0000,97.1%,100.0000,97.1%,Near-reference condition (97.1% of T...
1,Site 3 Karpa_kml (1)_0000,Natural Land Cover Proportion,1,37.6218,0.0000,—,—,—,Regional reference = 0 (no natural l...
2,Site 3 Karpa_kml (1)_0000,Landscape Connectivity (CPLAND),1,93.6702,1.0000,100.0%,—,—,Assessed via site value only — Tier ...
3,Site 3 Karpa_kml (1)_0000,Habitat Loss Rate,1,0.0000,4.1667,100.0%,—,—,Insufficient data for interpretation
4,Site 3 Karpa_kml (1)_0000,KBA/IBA Overlap,1,0.0000,—,—,—,—,Assessed via Protocol A absolute thr...
5,Site 3 Karpa_kml (1)_0000,Vegetation Structure (NDVI),2,0.3084,0.3945,78.2%,0.6216,49.6%,Severely degraded (49.6% of Tier 2 b...
6,Site 3 Karpa_kml (1)_0000,Habitat Health Index (HHI),2,2.4959,2.5320,98.6%,2.2137,100.0%,Near-reference condition (100.0% of ...
7,Site 3 Karpa_kml (1)_0000,Forest Landscape Integrity Index,2,4.9663,9.9647,49.8%,9.9636,49.8%,Severely degraded (49.8% of Tier 2 b...
8,Site 3 Karpa_kml (1)_0000,Ecosystem Integrity Index,2,0.2986,0.2871,100.0%,0.7164,41.7%,Severely degraded (41.7% of Tier 2 b...
9,Site 3 Karpa_kml (1)_0000,EII: Structural Integrity,2,0.3911,0.3420,100.0%,0.7593,51.5%,Degraded (51.5% of Tier 2 benchmark)


## 7. State of Nature Scoring

Converts intactness ratios into concern levels and computes the State of Nature Score per **Darukaa SoN Module PRD v2.0** and **TNFD LEAP Annex 2**.

**Three-tier scoring:**
- **Tier 1 (per-metric):** Intactness % → concern level (VL/L/M/H/VH) via threshold protocol
- **Tier 2 (per-dimension):** Mean concern numeric per TNFD dimension
- **Tier 3 (site-level):** `SoN_Score = (Σ dim_scores − n_dims) / (n_dims × 4) × 10` → 0–10 scale

**Protocol B v1.0 — Fixed, version-controlled thresholds (applied to intactness %):**
```
≥ 85% → VL (Very Low)    Basis: Newbold et al. 2016 planetary boundary (~90% BII)
70–84% → L  (Low)        Noticeably impacted, recoverable (SBTN science)
50–69% → M  (Moderate)   Substantially degraded — material loss (GLOBIO4 literature)
30–49% → H  (High)       Severely impaired
<  30% → VH (Very High)  Critically impaired
```
Review trigger: ≥10 Darukaa sites across ≥3 ecoregion types. These thresholds are fixed, not project-specific.

**Protocol A:** BII (NHM), FLII (Potapov), CERI (Butchart 2007), STAR_T (IUCN), KBA (TNFD/IBAT), Flagship (HSI literature), plant species richness (Protocol C / Tier 1 count comparison).

**Protocol D (Aquatic/eDNA):** Site-specific indicators (tspi, sabf, wcpi, wsdi, hsas, edpp, mspl, rci, riparian_ndvi_trend, jrc_water_persistence, shdi) are scored against absolute published thresholds for their specific physical quantity. No Tier 2 spatial reference — these require within-site temporal comparison or published ecological benchmarks.


**Dimension scoring:** Computed from all indicators with a populated concern level — **no minimum metric constraints** at this stage. Missing metrics are shown as `n=X of Y` so confidence is transparent. The SoN PRD v2.0 data sufficiency rules will be enforced in the product UI layer once in-situ data is integrated.

**Threats** (ghm, light pollution, HDI, LST, sdi, stsi, iri, ivsi) are contextual only — not in SoN score per TNFD Annex 2.

In [ ]:
import numpy as np

# ── Threshold Tables ─────────────────────────────────────────────────────
# Numeric encoding: VL=1, L=2, M=3, H=4, VH=5

# Protocol A: Published literature thresholds (applied to raw site value)
PROTOCOL_A = {
    # ── Ecosystem Condition ──
    "bii":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    "flii": {"breaks": [2.0,  4.0,  6.0,  8.0],  "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    "msa":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # Protocol D — Aquatic absolute thresholds
    # TSPI (NDCI): <-0.1=VL (oligotrophic), -0.1–0.0=L, 0.0–0.1=M (mesotrophic),
    #              0.1–0.2=H (eutrophic), >0.2=VH (hypereutrophic)
    "tspi": {"breaks": [-0.10, 0.00, 0.10, 0.20], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    # SABF: <0.05=VL, 0.05-0.15=L, 0.15-0.30=M, 0.30-0.50=H, >0.50=VH
    "sabf": {"breaks": [0.05, 0.15, 0.30, 0.50], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    # WCPI (clarity): >0.8=VL, 0.6-0.8=L, 0.4-0.6=M, 0.2-0.4=H, <0.2=VH
    "wcpi": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # WSDI (dynamics): <0.2=VL (stable), 0.2-0.4=L, 0.4-0.6=M (transitional),
    #                  0.6-0.8=H, >0.8=VH (highly unstable)
    "wsdi": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    # HSAS: >0.8=VL, 0.6-0.8=L, 0.4-0.6=M, 0.2-0.4=H, <0.2=VH
    "hsas": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # EDPP: >0.7=VL (good persistence), 0.5-0.7=L, 0.3-0.5=M, 0.1-0.3=H, <0.1=VH
    "edpp": {"breaks": [0.10, 0.30, 0.50, 0.70], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # MSPL: <0.2=VL, 0.2-0.4=L, 0.4-0.6=M, 0.6-0.8=H, >0.8=VH (high microbial stress)
    "mspl": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    # RCI: >0.7=VL, 0.5-0.7=L, 0.3-0.5=M, 0.1-0.3=H, <0.1=VH (poor riparian complexity)
    "rci":  {"breaks": [0.10, 0.30, 0.50, 0.70], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # riparian_ndvi_trend: >0.02/yr=VL (greening), 0.00-0.02=L, -0.02-0.00=M,
    #                      -0.05--0.02=H, <-0.05=VH (rapid browning)
    "riparian_ndvi_trend": {"breaks": [-0.05, -0.02, 0.00, 0.02], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # jrc_water_persistence: >0.8=VL, 0.6-0.8=L, 0.4-0.6=M, 0.2-0.4=H, <0.2=VH
    "jrc_water_persistence": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # shdi (morphometric): <1.5=VL (simple/circular), 1.5-2.0=L, 2.0-3.0=M,
    #                      3.0-5.0=H, >5.0=VH (very complex)
    # Note: higher shdi = more habitat complexity = ecologically BETTER for biodiversity
    # but scored VH here as it indicates habitat distinctiveness / monitoring priority
    # Rationale: shdi ≥ 5.0 = significantly irregular shoreline, high edge-habitat value
    "shdi": {"breaks": [1.5, 2.0, 3.0, 5.0], "scores": [1, 2, 3, 4, 5], "higher_is_better": True},
    # ── Species Population Size ──
    "flagship_habitat": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # ── Species Extinction Risk ──
    "ceri":   {"breaks": [0.10, 0.20, 0.35, 0.50], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    "star_t": {"breaks": [1.0,  3.0,  6.0,  9.0],  "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
    "kba_overlap": {"breaks": [1.0, 25.0, 75.0, 99.9], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
}

# Protocol B v1.0 — applied to Tier 2 intactness % (or Tier 1 where T2 unavailable)
PROTOCOL_B_BREAKS = [30, 50, 70, 85]
PROTOCOL_B_SCORES = [5,  4,  3,  2, 1]

# Protocol C: Tier 1 regional comparison for count/connectivity indicators
PROTOCOL_C_INDICATORS = {"cpland", "endemic_richness", "threatened_richness",
                          "endemic_plant_richness", "threatened_plant_richness"}

# TNFD Annex 2 dimension membership — all 44 indicators
DIM_MAP = {
    1: ["natural_habitat", "natural_landcover", "cpland", "forest_loss_rate", "kba_overlap"],
    2: ["ndvi", "habitat_health", "flii", "eii", "eii_structural",
        "eii_compositional", "eii_functional", "bii", "pdf", "aridity_index",
        # v6.4 aquatic/eDNA
        "tspi", "sabf", "wcpi", "wsdi", "hsas", "edpp", "mspl", "rci",
        # v6.5 terrestrial vegetation
        "riparian_ndvi_trend", "jrc_water_persistence", "shdi", "lai", "chm"],
    3: ["endemic_richness", "flagship_habitat", "endemic_plant_richness"],
    4: ["threatened_richness", "ceri", "star_t", "threatened_plant_richness"],
}
THREATS = ["ghm", "light_pollution", "hdi", "lst_day", "lst_night",
           "sdi", "stsi", "iri", "ivsi"]  # pillar 5 — contextual only
DIM_NAMES = {
    1: "Ecosystem Extent",
    2: "Ecosystem Condition",
    3: "Species Population Size",
    4: "Species Extinction Risk",
}
PROTOCOL_A_INDICATORS = set(PROTOCOL_A.keys())


# ── Helpers ──────────────────────────────────────────────────────────────

def _is_valid(v):
    if v is None: return False
    try: return np.isfinite(float(v))
    except: return False

def apply_protocol_a(indicator_name, site_value):
    if not _is_valid(site_value): return None
    spec = PROTOCOL_A[indicator_name]
    val = float(site_value)
    for i, b in enumerate(spec["breaks"]):
        if val < b: return spec["scores"][i]
    return spec["scores"][-1]

def apply_protocol_b(intactness_ratio):
    if not _is_valid(intactness_ratio): return None
    pct = float(intactness_ratio) * 100
    for i, b in enumerate(PROTOCOL_B_BREAKS):
        if pct < b: return PROTOCOL_B_SCORES[i]
    return PROTOCOL_B_SCORES[-1]

def concern_label(numeric):
    if numeric is None: return "—"
    return {1: "VL", 2: "L", 3: "M", 4: "H", 5: "VH"}.get(round(numeric), "—")

def dim_label(score):
    if score is None: return "—"
    s = round(score * 2) / 2
    if s <= 1.5: return "Very Low"
    if s <= 2.5: return "Low"
    if s <= 3.5: return "Moderate"
    if s <= 4.5: return "High"
    return "Very High"


# ── Build results lookup ──────────────────────────────────────────────────
results_lookup = {}
for _, row in df.iterrows():
    name = row.get("indicator_name") or row.get("name")
    if name is None:
        for spec in registry.all():
            if spec.display_name == row.get("display_name"):
                name = spec.name; break
    if name:
        results_lookup[name] = row.to_dict()


# ── Per-metric concern assignment ────────────────────────────────────────
metric_concerns = {}

for name, row in results_lookup.items():
    if name in THREATS:
        continue

    site_val = row.get("site_value")
    t2       = row.get("tier2_intactness")
    t1       = row.get("tier1_intactness")

    if not _is_valid(site_val): site_val = None
    if not _is_valid(t2):       t2 = None
    if not _is_valid(t1):       t1 = None

    spec_obj = registry.get(name) if name in registry else None

    if name in PROTOCOL_A_INDICATORS and site_val is not None:
        cn = apply_protocol_a(name, site_val)
        protocol = "A (published absolute threshold)"
        intactness_used = None

    elif t2 is not None:
        cn = apply_protocol_b(t2)
        protocol = "B v1.0 (Tier 2 intactness)"
        intactness_used = t2

    elif t1 is not None:
        cn = apply_protocol_b(t1)
        if name in PROTOCOL_C_INDICATORS:
            protocol = "C (Tier 1 regional — designed ref for this indicator)"
        else:
            protocol = "B v1.0 (Tier 1 fallback — T2 unavailable)"
        intactness_used = t1

    else:
        cn = None
        protocol = "— (no reference computable)"
        intactness_used = None

    metric_concerns[name] = {
        "concern_numeric":  cn,
        "concern_label":    concern_label(cn),
        "protocol":         protocol,
        "intactness_used":  intactness_used,
        "site_value":       site_val,
        "display_name":     row.get("display_name", name),
    }

n_scored = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is not None)
n_nodata = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is None)
n_prot_a = sum(1 for v in metric_concerns.values() if "A" in v["protocol"])
n_prot_b = sum(1 for v in metric_concerns.values() if "B v1.0" in v["protocol"] and "C" not in v["protocol"])
n_prot_c = sum(1 for v in metric_concerns.values() if "C" in v["protocol"])
print(f"✓ Per-metric concern levels assigned")
print(f"  Protocol A/D (published):  {n_prot_a}")
print(f"  Protocol B v1.0 (T2):      {n_prot_b}")
print(f"  Protocol C / B-T1:         {n_prot_c}")
print(f"  No reference:              {n_nodata}")
print(f"  Total scored:              {n_scored} / {len(metric_concerns)}")

✓ Per-metric concern levels assigned
  Protocol A (published):    6
  Protocol B v1.0 (T2):      10
  Protocol C / B-T1:         2
  No reference:              2
  Total scored:              18 / 20


In [ ]:
# ── Per-metric concern table ─────────────────────────────────────────────
print(f"\n{'='*96}")
print(f"PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0 / A / C / D)")
print(f"{'='*96}")
print(f"  {'Dim':<5} {'Indicator':<45} {'Site Value':<12} {'Intactness':<12} {'Concern':<8} Protocol")
print("  " + "-"*94)

for dim_num, indicators in DIM_MAP.items():
    print(f"\n  ── Dim {dim_num}: {DIM_NAMES[dim_num]} ──")
    for name in indicators:
        mc  = metric_concerns.get(name)
        row = results_lookup.get(name, {})
        dname = (mc["display_name"] if mc else row.get("display_name", name))[:44]

        if mc is None:
            print(f"  {dim_num:<5} {dname:<45} {'—':<12} {'—':<12} {'—':<8} not run")
            continue

        sv = mc["site_value"]
        sv_str = f"{sv:.4f}" if sv is not None else "—"

        iu = mc["intactness_used"]
        if iu is not None:
            intact_str = f"{iu:.1%}"
        elif "A" in mc["protocol"] or "D" in mc["protocol"]:
            intact_str = "n/a (raw)"
        else:
            intact_str = "no ref"

        print(f"  {dim_num:<5} {dname:<45} {sv_str:<12} {intact_str:<12} {mc['concern_label']:<8} {mc['protocol']}")

print(f"\n  ── Threats (contextual — not in SoN score) ──")
for name in THREATS:
    row   = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:44]
    sv    = row.get("site_value")
    t1    = row.get("tier1_intactness")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    t1_str = f"{float(t1):.1%}"  if _is_valid(t1) else "—"
    print(f"  {'T':<5} {dname:<45} {sv_str:<12} {t1_str:<12} {'—':<8} Tier 1 contextual")


PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Indicator                                  Site Value   Intactness   Concern  Protocol
  ------------------------------------------------------------------------------------------

  ── Dim 1: Ecosystem Extent ──
  1     Natural Habitat Extent                     97.0667      97.1%        VL       B v1.0 (Tier 2 intactness)
  1     Natural Land Cover Proportion              37.6218      no ref       —        — (no reference computable)
  1     Landscape Connectivity (CPLAND)            93.6702      100.0%       VL       C (Tier 1 regional — designed ref for this indicator)
  1     Habitat Loss Rate                          0.0000       100.0%       VL       B v1.0 (Tier 1 fallback — T2 unavailable)
  1     KBA/IBA Overlap                            0.0000       no ref       VL       A (published absolute threshold)

  ── Dim 2: Ecosystem Condition ──
  2     Vegetation Structure (NDVI)                0.3084       49.6

In [ ]:
# ── Per-metric concern table ─────────────────────────────────────────────
print(f"\n{'='*96}")
print(f"PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0 / A / C / D)")
print(f"{'='*96}")
print(f"  {'Dim':<5} {'Indicator':<45} {'Site Value':<12} {'Intactness':<12} {'Concern':<8} Protocol")
print("  " + "-"*94)

for dim_num, indicators in DIM_MAP.items():
    print(f"\n  ── Dim {dim_num}: {DIM_NAMES[dim_num]} ──")
    for name in indicators:
        mc  = metric_concerns.get(name)
        row = results_lookup.get(name, {})
        dname = (mc["display_name"] if mc else row.get("display_name", name))[:44]

        if mc is None:
            print(f"  {dim_num:<5} {dname:<45} {'—':<12} {'—':<12} {'—':<8} not run")
            continue

        sv = mc["site_value"]
        sv_str = f"{sv:.4f}" if sv is not None else "—"

        iu = mc["intactness_used"]
        if iu is not None:
            intact_str = f"{iu:.1%}"
        elif "A" in mc["protocol"] or "D" in mc["protocol"]:
            intact_str = "n/a (raw)"
        else:
            intact_str = "no ref"

        print(f"  {dim_num:<5} {dname:<45} {sv_str:<12} {intact_str:<12} {mc['concern_label']:<8} {mc['protocol']}")

print(f"\n  ── Threats (contextual — not in SoN score) ──")
for name in THREATS:
    row   = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:44]
    sv    = row.get("site_value")
    t1    = row.get("tier1_intactness")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    t1_str = f"{float(t1):.1%}"  if _is_valid(t1) else "—"
    print(f"  {'T':<5} {dname:<45} {sv_str:<12} {t1_str:<12} {'—':<8} Tier 1 contextual")


DIMENSION SCORES  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Dimension                      Score   Concern         Coverage
  ----------------------------------------------------------------------
  1     Ecosystem Extent               1.00    Very Low        4/5  ← 1 metric(s) could not be computed
  2     Ecosystem Condition            2.90    Moderate        10/10
  3     Species Population Size        4.00    High            1/2  ← 1 metric(s) could not be computed
  4     Species Extinction Risk        1.00    Very Low        3/3

STATE OF NATURE SCORE  (SoN PRD v2.0)
  SoN Score:   3.1 / 10
  Concern:     Very Low
  Dims used:   4 of 4

  Formula: (8.90 - 4) / (4 x 4) x 10 = 3.1

  Protocol B v1.0: >=85%=VL | 70-84%=L | 50-69%=M | 30-49%=H | <30%=VH
  Basis: Newbold et al. (2016) + GLOBIO4/SBTN literature
  Review trigger: >=10 Darukaa sites across >=3 ecoregion types

TRAJECTORY: BASELINE RUN (Year 0)
  ✓ Baseline saved → ./output/baseline_scorecard.csv
  Set IS_BASELINE_RUN = F

In [ ]:
# ── Dimension concern profile ─────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"DIMENSION CONCERN PROFILES")
print(f"{'='*70}")
print(f"  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High\n")

labels_order = ["VL", "L", "M", "H", "VH", "—"]

for dim_num, indicators in DIM_MAP.items():
    counts = {l: [] for l in labels_order}
    for name in indicators:
        mc = metric_concerns.get(name)
        cl = mc["concern_label"] if mc else "—"
        if cl not in counts: cl = "—"
        counts[cl].append(mc["display_name"] if mc else name)
    print(f"  Dim {dim_num} — {DIM_NAMES[dim_num]}")
    for level in labels_order:
        items = counts[level]
        if not items: continue
        print(f"    {level:>2}: {len(items)} — {', '.join(items)}")
    print()

# ── Threat assessment ─────────────────────────────────────────────────────────
THREAT_THRESHOLDS = {
    "ghm": {"breaks": [0.10, 0.30, 0.60, 0.90],
            "labels": ["Very Low","Low","Moderate","High","Very High"],
            "unit": "(0-1)", "source": "Theobald et al. 2025"},
    "light_pollution": {"breaks": [1.0, 5.0, 30.0, 100.0],
                        "labels": ["Very Low","Low","Moderate","High","Very High"],
                        "unit": "nW·cm⁻²·sr⁻¹", "source": "NASA Black Marble / VIIRS DNB"},
    "hdi": {"breaks": [0.50, 0.60, 0.70, 0.80],
            "labels": ["Very Low","Low","Moderate","High","Very High"],
            "unit": "(0-1)", "source": "Mishra et al. 2017"},
    "lst_night": {"breaks": [22.0, 26.0, 30.0, 34.0],
                  "labels": ["Very Low","Low","Moderate","High","Very High"],
                  "unit": "°C", "source": "KMC et al. 2025"},
    "lst_day": {"breaks": [32.0, 36.0, 40.0, 44.0],
                "labels": ["Very Low","Low","Moderate","High","Very High"],
                "unit": "°C", "source": "Muse, Clement & Mach 2024"},
    # v6.4 aquatic threats — site-relative, no published absolute thresholds
    # Reported as raw values with qualitative band descriptions
    "sdi":  {"breaks": [0.10, 0.25, 0.50, 0.75],
             "labels": ["Very Low","Low","Moderate","High","Very High"],
             "unit": "fraction", "source": "Allan et al. (2002)"},
    "stsi": {"breaks": [0.20, 0.40, 0.60, 0.80],
             "labels": ["Very Low","Low","Moderate","High","Very High"],
             "unit": "(0-1, site-rel.)", "source": "Jimenez-Munoz et al. (2014)"},
    "iri":  {"breaks": [0.20, 0.40, 0.60, 0.80],
             "labels": ["Very Low","Low","Moderate","High","Very High"],
             "unit": "(0-1)", "source": "Bellard et al. (2016)"},
    # v6.5
    "ivsi": {"breaks": [0.05, 0.15, 0.30, 0.50],
             "labels": ["Very Low","Low","Moderate","High","Very High"],
             "unit": "fraction", "source": "Paz-Kagan et al. (2019)"},
}

def threat_label(name, site_val):
    if not _is_valid(site_val): return "—"
    spec = THREAT_THRESHOLDS.get(name)
    if spec is None: return "—"
    val = float(site_val)
    for b, label in zip(spec["breaks"], spec["labels"]):
        if val < b: return label
    return spec["labels"][-1]

print(f"\n{'='*70}")
print(f"THREATS & PRESSURES — Categorical Assessment")
print(f"(Contextual layer — not included in SoN score)")
print(f"{'='*70}")
print(f"  {'Indicator':<38} {'Value':<12} {'Unit':<16} Assessment")
print("  " + "-"*75)

for name in THREATS:
    row = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:37]
    sv  = row.get("site_value")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    spec = THREAT_THRESHOLDS.get(name, {})
    unit = spec.get("unit", "")
    label = threat_label(name, sv)
    print(f"  {dname:<38} {sv_str:<12} {unit:<16} {label}")


DIMENSION CONCERN PROFILES
(count of indicators at each concern level within each dimension)
  This is the primary output for TNFD LEAP Step 3 disclosure.
  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High

  Dim 1 — Ecosystem Extent
    VL: 4 — Natural Habitat Extent, Landscape Connectivity (CPLAND), Habitat Loss Rate, KBA/IBA Overlap
     —: 1 — Natural Land Cover Proportion

  Dim 2 — Ecosystem Condition
    VL: 2 — Habitat Health Index (HHI), Aridity Index
     M: 5 — Forest Landscape Integrity Index, EII: Structural Integrity, EII: Compositional Integrity, EII: Functional Integrity, Biodiversity Intactness Index
     H: 3 — Vegetation Structure (NDVI), Ecosystem Integrity Index, Potentially Disappeared Fraction

  Dim 3 — Species Population Size
     H: 1 — Flagship Habitat Viability
     —: 1 — Endemic Species Richness

  Dim 4 — Species Extinction Risk
    VL: 3 — Threatened Species Richness, Composite Extinction-Risk Index, STAR_T (Threat Abatement)


THREATS

In [ ]:
# ── Species Lists for Reporting ─────────────────────────────────────────
import ee
from darukaa_reference.indicators import (
    extract_threatened_richness,
    extract_ceri,
    extract_endemic_richness,
    extract_endemic_plant_richness,
    extract_threatened_plant_richness,
)

def _kml_to_ee_geometries(kml_path):
    import os
    from shapely.geometry import mapping
    from shapely.ops import transform as shapely_transform
    geoms = {}
    if kml_path.lower().endswith('.kmz'):
        import zipfile
        with zipfile.ZipFile(kml_path, 'r') as z:
            kml_names = [n for n in z.namelist() if n.lower().endswith('.kml')]
            kml_bytes = z.read(kml_names[0])
        kml_str = kml_bytes.decode('utf-8', errors='replace')
    else:
        with open(kml_path, 'rb') as f:
            kml_str = f.read().decode('utf-8', errors='replace')
    try:
        from fastkml import kml as fkml
        k = fkml.KML(); k.from_string(kml_str.encode())
        features = list(list(k.features())[0].features())
        for i, feat in enumerate(features):
            name = getattr(feat, 'name', None) or f"site_{i:04d}"
            geom = feat.geometry
            if geom is None: continue
            if geom.has_z:
                geom = shapely_transform(lambda x,y,z=None:(x,y), geom)
            geoms[f"{name}_{i:04d}"] = ee.Geometry(mapping(geom))
    except Exception:
        from lxml import etree
        from shapely.geometry import Polygon
        root = etree.fromstring(kml_str.encode())
        ns = {'kml': 'http://www.opengis.net/kml/2.2'}
        placemarks = root.findall('.//kml:Placemark', ns)
        for i, pm in enumerate(placemarks):
            name_el = pm.find('kml:name', ns)
            name = name_el.text if name_el is not None else f"site_{i:04d}"
            coords_el = pm.find('.//kml:coordinates', ns)
            if coords_el is None: continue
            coords_text = coords_el.text.strip()
            coords = []
            for c in coords_text.split():
                parts = c.split(',')
                if len(parts) >= 2: coords.append((float(parts[0]), float(parts[1])))
            if len(coords) >= 3:
                geom = Polygon(coords)
                geoms[f"{name}_{i:04d}"] = ee.Geometry(mapping(geom))
    return geoms

print("Parsing site geometry from KML...")
try:
    site_geoms = _kml_to_ee_geometries(SITE_FILE)
    print(f"✓ Found {len(site_geoms)} site(s): {list(site_geoms.keys())}")
except Exception as e:
    print(f"✗ KML parse failed: {e}"); site_geoms = {}

SPECIES_INDICATORS = {
    "threatened_richness":       ("Threatened Species (CR/EN/VU) — mammals + birds", extract_threatened_richness),
    "ceri":                      ("CERI Species — all IUCN categories",               extract_ceri),
    "endemic_richness":          ("Endemic Species (range <100k km²) — mammals+birds", extract_endemic_richness),
    "endemic_plant_richness":    ("Endemic Plants (range <100k km²)",                 extract_endemic_plant_richness),
    "threatened_plant_richness": ("Threatened Plants (CR/EN/VU)",                     extract_threatened_plant_richness),
}

print(f"\n{'='*70}")
print(f"SPECIES LISTS (for reporting transparency)")
print(f"{'='*70}")
all_species_data = {}

for site_id, geom in site_geoms.items():
    print(f"\n  Site: {site_id}")
    site_species = {}
    for ind_name, (label, extract_fn) in SPECIES_INDICATORS.items():
        print(f"\n  {label}")
        try:
            result = extract_fn(geom, config)
            meta = result.get("metadata", {})
            species_list = meta.get("species_list", [])
            if not species_list:
                print(f"    No species returned"); continue
            # Animals have 'group' field; plants do not
            mammals = sorted([s for s in species_list if s.get("group") == "mammal"],
                             key=lambda x: x.get("category",""))
            birds   = sorted([s for s in species_list if s.get("group") == "bird"],
                             key=lambda x: x.get("category",""))
            plants  = [s for s in species_list if "group" not in s]
            if mammals:
                print(f"\n    Mammals ({len(mammals)}):")
                for s in mammals: print(f"      [{s.get('category','?'):2s}] {s['name']}")
            if birds:
                print(f"\n    Birds ({len(birds)}):")
                for s in birds:   print(f"      [{s.get('category','?'):2s}] {s['name']}")
            if plants:
                print(f"\n    Plants ({len(plants)}):")
                for s in plants:  print(f"      [{s.get('category','?'):2s}] {s.get('name','?')}")
            site_species[ind_name] = species_list
        except Exception as e:
            print(f"    Error running {ind_name}: {e}")
    all_species_data[site_id] = site_species

import json as _json
species_path = f"{OUTPUT_DIR}/species_lists.json"
with open(species_path, "w") as _f:
    _json.dump(all_species_data, _f, indent=2)
print(f"\n✓ Species lists saved → {species_path}")

Parsing site geometry from KML...
✓ Found 1 site(s): ['TCF PLANTATION KARPA PF384 Area 150 hqtr._0000']

SPECIES LISTS (for reporting transparency)
  Source: IUCN Red List range maps (mammals + birds)
  CR=Critically Endangered, EN=Endangered, VU=Vulnerable, NT=Near Threatened

  Site: TCF PLANTATION KARPA PF384 Area 150 hqtr._0000

  Threatened Species (CR/EN/VU) — mammals + birds

    Mammals (9):
      [EN] Panthera tigris
      [EN] Manis crassicaudata
      [EN] Cuon alpinus
      [VU] Acinonyx jubatus
      [VU] Tetracerus quadricornis
      [VU] Rusa unicolor
      [VU] Panthera leo
      [VU] Melursus ursinus
      [VU] Panthera pardus

    Birds (19):
      [CR] Sypheotides indicus
      [CR] Sarcogyps calvus
      [CR] Gyps bengalensis
      [CR] Gyps indicus
      [EN] Sterna acuticauda
      [EN] Rynchops albicollis
      [EN] Neophron percnopterus
      [EN] Aquila nipalensis
      [EN] Calidris tenuirostris
      [VU] Grus antigone
      [VU] Aquila heliaca
      [VU] Cal

## 8. Download results

In [ ]:
import os, json as _json
from google.colab import files

son_output = {
    "metric_concerns": metric_concerns,
    "dim_scores": {
        str(d): {
            "score":         dim_scores.get(d),
            "concern_label": dim_label(dim_scores.get(d)),
            "n_populated":   len(dim_populated.get(d, [])),
            "n_total":       len(DIM_MAP[d]),
        }
        for d in [1, 2, 3, 4]
    },
    "son_score":   son_score,
    "son_concern": son_concern_label(son_score),
    "n_dims_used": n_valid,
    "partial_score": partial,
    "threshold_protocol": "Protocol A/D (published) + Protocol B v1.0 (Newbold 2016 + GLOBIO4/SBTN) + Protocol C (Tier 1 count)",
    "formula": "SoN = (Sigma dim_scores - n_dims) / (n_dims x 4) x 10",
    "is_baseline_run": IS_BASELINE_RUN,
    "project_type": PROJECT_TYPE,
    "indicator_version": "v6.5 (44 indicators)",
}

combined_path = f"{OUTPUT_DIR}/baseline_scorecard_with_son.json"
with open(combined_path, "w") as _f:
    _json.dump({**report, "son_scoring": son_output}, _f, indent=2, default=str)

if IS_BASELINE_RUN and os.path.exists(BASELINE_FILE):
    files.download(BASELINE_FILE)
if not IS_BASELINE_RUN:
    try:
        traj_path = f"{OUTPUT_DIR}/trajectory.csv"
        df_traj.to_csv(traj_path, index=False)
        files.download(traj_path)
    except NameError:
        pass

print("Downloading results...")
files.download(f"{OUTPUT_DIR}/benchmark_scorecard.csv")
files.download(combined_path)
try: files.download(species_path)
except NameError: pass

print("\n✓ Downloaded:")
print("  benchmark_scorecard.csv           — raw benchmark results")
print("  baseline_scorecard_with_son.json  — includes SoN concern levels & score")
if IS_BASELINE_RUN:
    print("  baseline_scorecard.csv            — Year 0 baseline for trajectory tracking")
if not IS_BASELINE_RUN:
    print("  trajectory.csv                    — Year 1+ trajectory labels")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Downloaded:
  baseline_scorecard.csv           — same, CSV format
  baseline_scorecard_with_son.json — includes SoN concern levels & score
  baseline_scorecard.csv             — Year 0 baseline for trajectory tracking


---

## (Optional) Add a custom indicator

Adding any new indicator requires only a few lines — no pipeline changes needed.

In [ ]:
def extract_my_indicator(geometry, config):
    """
    Your extraction logic here.
    Return dict with 'value' (float) and 'pixels' (array or None).
    """
    return {"value": 0.42, "pixels": None}

registry.register(
    name="my_indicator",
    display_name="My Custom Indicator",
    source_type="gee",
    extract_fn=extract_my_indicator,
    unit="index",
    value_range=(0.0, 1.0),
    citation="Author et al. (year). DOI:...",
    tier2_eligible=True,
    higher_is_better=True,
    reference_radius_km=50.0,
    pillar=2,
)

print(f"✓ Registry now has {len(registry)} indicators")
print(f"  Tier 2 eligible: {len(registry.tier2_indicators())}")